In [ ]:
import json
import time
import datetime
import requests
import pandas as pd
import os
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import country_converter as coco

## Import Data

In [ ]:
# Confirm current working directory
cwd = os.getcwd()
print(f"Current working directory: {cwd}")

In [ ]:
fpu_raw = f"{cwd}/outputs/FPU_Factiva_raw.xlsx"

In [ ]:
# Variables expected from the revised Factiva extraction notebook
# The monthly sheet should contain the aggregate FPU counts plus the three category counts.
fpu_vars = [
    "publication_datetime",
    "numerator_base_count",
    "denominator_count",
    "numerator_rece_count",
    "tax_count",
    "expenditure_count",
    "debt_count",
    "iso3"
]


In [ ]:
# Read revised monthly Factiva output.
# The collector now writes tax_count, expenditure_count, and debt_count directly
# from three category-specific Factiva count queries, while numerator_base_count
# comes from the aggregate (Tax OR Expenditure OR Debt) query.
df_fpu_raw = pd.read_excel(fpu_raw, sheet_name="monthly")

missing_cols = [c for c in fpu_vars if c not in df_fpu_raw.columns]
if missing_cols:
    raise ValueError(
        "The monthly input is missing required columns: " + ", ".join(missing_cols) +
        ". Run the revised 01_FPU_Factiva_Tax_Expenditure_Debt_MasterOutput.ipynb "
        "first so the monthly sheet contains tax_count, expenditure_count, and debt_count."
    )

df_fpu_raw = df_fpu_raw[fpu_vars].copy()
df_fpu_raw['publication_datetime'] = pd.to_datetime(df_fpu_raw['publication_datetime'])

count_cols = [
    'numerator_base_count', 'numerator_rece_count',
    'tax_count', 'expenditure_count', 'debt_count', 'denominator_count'
]
df_fpu_raw[count_cols] = df_fpu_raw[count_cols].apply(pd.to_numeric, errors='coerce').fillna(0)

print("Connected monthly input columns confirmed:")
print(fpu_vars)


In [ ]:
# Pipeline QC: verify aggregate and subcategory count logic.
# Because categories overlap, tax + expenditure + debt can be greater than aggregate.
_pipeline_qc = df_fpu_raw[[
    'iso3', 'publication_datetime', 'numerator_base_count',
    'tax_count', 'expenditure_count', 'debt_count', 'denominator_count'
]].copy()
_pipeline_qc['subcategory_sum'] = _pipeline_qc[['tax_count', 'expenditure_count', 'debt_count']].sum(axis=1)
_pipeline_qc['subcategory_sum_minus_aggregate'] = (
    _pipeline_qc['subcategory_sum'] - _pipeline_qc['numerator_base_count']
)

# A category-specific count should not exceed the aggregate OR count for the same country-month.
for c in ['tax_count', 'expenditure_count', 'debt_count']:
    bad = (_pipeline_qc[c] > _pipeline_qc['numerator_base_count']).sum()
    if bad:
        raise ValueError(f"QC failed: {c} exceeds numerator_base_count in {bad} rows.")

print("Pipeline QC passed: every category count <= aggregate OR count.")
print("Rows with intentional category overlap:",
      (_pipeline_qc['subcategory_sum_minus_aggregate'] > 0).sum())


## Truncate Data from 1995 onwards

In [ ]:
# keep everything on or after Jan 1, 1995 to follow the IMF paper for period analysis since a large number of countries not having good coverage before that
final_df = df_fpu_raw[
    df_fpu_raw['publication_datetime'] >= pd.Timestamp('1995-01-01')
].copy()

In [ ]:
# ============================================================
# COUNTRY-LEVEL FPU INDICES
# Aggregate + recession-excluded + Tax + Expenditure + Debt
# ============================================================

# 1. Compute raw ratios using the SAME denominator for all indices
final_df['FPU_base_raw'] = (
    final_df['numerator_base_count'] / final_df['denominator_count']
)

final_df['FPU_excl_rec_raw'] = (
    final_df['numerator_rece_count'] / final_df['denominator_count']
)

final_df['Tax_FPU_raw'] = (
    final_df['tax_count'] / final_df['denominator_count']
)

final_df['Expenditure_FPU_raw'] = (
    final_df['expenditure_count'] / final_df['denominator_count']
)

final_df['Debt_FPU_raw'] = (
    final_df['debt_count'] / final_df['denominator_count']
)

# Avoid inf values when denominator_count is zero
raw_vars = [
    'FPU_base_raw',
    'FPU_excl_rec_raw',
    'Tax_FPU_raw',
    'Expenditure_FPU_raw',
    'Debt_FPU_raw'
]
final_df[raw_vars] = final_df[raw_vars].replace([float('inf'), -float('inf')], pd.NA)

# 2. Normalize within each country:
#    transformed index = (raw - country mean) / country std + 100
# This keeps the same normalization convention as the original notebook.
def normalize_within_country(series):
    sd = series.std(ddof=0)
    if pd.isna(sd) or sd == 0:
        return pd.Series(pd.NA, index=series.index, dtype='Float64')
    return (series - series.mean()) / sd + 100

normalization_map = {
    'FPU_base_raw': 'FPU_base_transform',
    'FPU_excl_rec_raw': 'FPU_excl_rec_transform',
    'Tax_FPU_raw': 'Tax_FPU_transform',
    'Expenditure_FPU_raw': 'Expenditure_FPU_transform',
    'Debt_FPU_raw': 'Debt_FPU_transform'
}

for raw_var, transformed_var in normalization_map.items():
    final_df[transformed_var] = (
        final_df.groupby('iso3')[raw_var]
                .transform(normalize_within_country)
    )


In [ ]:
# Compute mean of denominator by iso3
denom_avg = final_df.groupby('iso3', dropna=False)['denominator_count'].mean().reset_index().rename(columns={'denominator_count':'denom_avg'})

# Merge denom_avg back to the raw data
df = final_df.merge(denom_avg, on='iso3', how='left')

# Show summary statistics for denom_avg (similar to Stata's summarize, detail)
summary = denom_avg['denom_avg'].describe()
print('\nDenom_avg summary:')
print(summary)

# Also show standard deviation separately to mimic Stata's `summarize, detail` output
print(f"std: {denom_avg['denom_avg'].std():.4f}")

# Filter out countries with denom_avg < 50
cutoff = 50
good_isos = denom_avg.loc[denom_avg['denom_avg'] >= cutoff, 'iso3'].tolist()

In [ ]:
filtered_count = df['iso3'].isin(good_isos).sum()
print(f"Keeping {len(good_isos)} iso3 groups; {filtered_count} rows remain after filtering (denom_avg >= {cutoff}).")

# Create the filtered dataframe
df_fpu = df[df['iso3'].isin(good_isos)].copy()

# Optional: reset index
df_fpu.reset_index(drop=True, inplace=True)

### Global FPU

In [ ]:
print(df_fpu.columns)

# Quick check: the revised cleaned data should now contain all three category indices.
check_cols = [
    'Tax_FPU_raw', 'Tax_FPU_transform',
    'Expenditure_FPU_raw', 'Expenditure_FPU_transform',
    'Debt_FPU_raw', 'Debt_FPU_transform'
]
print("\nCategory FPU columns present:")
print({c: c in df_fpu.columns for c in check_cols})


In [ ]:
# ============================================================
# GLOBAL FPU INDICES
# ============================================================

# 1. Sum numerator/category counts and denominator across retained countries
count_vars = [
    'numerator_base_count',
    'numerator_rece_count',
    'tax_count',
    'expenditure_count',
    'debt_count',
    'denominator_count'
]

df_global = (
    df_fpu.groupby('publication_datetime', as_index=False)[count_vars]
          .sum()
)

# 2. Compute global raw indices using the same global denominator
# NOTE: tax_count + expenditure_count + debt_count may exceed numerator_base_count
# because the three category queries are intentionally non-mutually-exclusive.
df_global['FPU_base_raw'] = (
    df_global['numerator_base_count'] / df_global['denominator_count']
)

df_global['FPU_excl_rec_raw'] = (
    df_global['numerator_rece_count'] / df_global['denominator_count']
)

df_global['Tax_FPU_raw'] = (
    df_global['tax_count'] / df_global['denominator_count']
)

df_global['Expenditure_FPU_raw'] = (
    df_global['expenditure_count'] / df_global['denominator_count']
)

df_global['Debt_FPU_raw'] = (
    df_global['debt_count'] / df_global['denominator_count']
)

raw_vars_global = [
    'FPU_base_raw',
    'FPU_excl_rec_raw',
    'Tax_FPU_raw',
    'Expenditure_FPU_raw',
    'Debt_FPU_raw'
]
df_global[raw_vars_global] = df_global[raw_vars_global].replace(
    [float('inf'), -float('inf')], pd.NA
)


In [ ]:
# Normalize each GLOBAL FPU series: (value - full-sample mean) / std + 100

def normalize_global(series):
    sd = series.std(ddof=0)
    if pd.isna(sd) or sd == 0:
        return pd.Series(pd.NA, index=series.index, dtype='Float64')
    return (series - series.mean()) / sd + 100

global_normalization_map = {
    'FPU_base_raw': 'FPU_base_transform',
    'FPU_excl_rec_raw': 'FPU_excl_rec_transform',
    'Tax_FPU_raw': 'Tax_FPU_transform',
    'Expenditure_FPU_raw': 'Expenditure_FPU_transform',
    'Debt_FPU_raw': 'Debt_FPU_transform'
}

for raw_var, transformed_var in global_normalization_map.items():
    df_global[transformed_var] = normalize_global(df_global[raw_var])


In [ ]:
# Plot all five global indices separately for a clean diagnostic view
plot_series = [
    ('FPU_base_transform', 'Global Fiscal Policy Uncertainty – Baseline'),
    ('FPU_excl_rec_transform', 'Global Fiscal Policy Uncertainty – Excluding Recession'),
    ('Tax_FPU_transform', 'Global Tax Policy Uncertainty'),
    ('Expenditure_FPU_transform', 'Global Expenditure Policy Uncertainty'),
    ('Debt_FPU_transform', 'Global Debt Policy Uncertainty')
]

for var, title in plot_series:
    plt.figure(figsize=(12, 4))
    plt.plot(df_global['publication_datetime'], df_global[var])
    plt.xlabel('Publication Date')
    plt.ylabel('FPU Index')
    plt.title(title)
    plt.axhline(100, linestyle='--', linewidth=0.8)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## Export FPU Dataset

In [ ]:
# ============================================================
# BUILD MONTHLY MASTER DATASET
# ============================================================
# Keep counts, raw ratios, normalized indices, and overlap diagnostics together.

monthly_master = df_fpu.copy()

monthly_master['subcategory_sum'] = monthly_master[
    ['tax_count', 'expenditure_count', 'debt_count']
].sum(axis=1)
monthly_master['subcategory_sum_minus_aggregate'] = (
    monthly_master['subcategory_sum'] - monthly_master['numerator_base_count']
)

# Publication-friendly normalized names
monthly_master['FPU_base'] = monthly_master['FPU_base_transform']
monthly_master['FPU_excl_rec'] = monthly_master['FPU_excl_rec_transform']
monthly_master['Tax_FPU'] = monthly_master['Tax_FPU_transform']
monthly_master['Expenditure_FPU'] = monthly_master['Expenditure_FPU_transform']
monthly_master['Debt_FPU'] = monthly_master['Debt_FPU_transform']
monthly_master['publication_month'] = (
    pd.to_datetime(monthly_master['publication_datetime']).dt.to_period('M').astype(str)
)


In [ ]:
# ============================================================
# ADD COUNTRY NAMES AND ORDER MONTHLY MASTER COLUMNS
# ============================================================
cc = coco.CountryConverter()
monthly_master['country'] = cc.convert(names=monthly_master['iso3'], to='name_short')

monthly_master_cols = [
    'iso3', 'country', 'publication_month',
    'denominator_count',
    'numerator_base_count', 'numerator_rece_count',
    'tax_count', 'expenditure_count', 'debt_count',
    'subcategory_sum', 'subcategory_sum_minus_aggregate',
    'FPU_base_raw', 'FPU_excl_rec_raw',
    'Tax_FPU_raw', 'Expenditure_FPU_raw', 'Debt_FPU_raw',
    'FPU_base', 'FPU_excl_rec',
    'Tax_FPU', 'Expenditure_FPU', 'Debt_FPU',
    'denom_avg'
]
monthly_master = monthly_master[[c for c in monthly_master_cols if c in monthly_master.columns]]
monthly_master = monthly_master.sort_values(['iso3', 'publication_month']).reset_index(drop=True)


In [ ]:
# ============================================================
# BUILD QUARTERLY MASTER DATASET FROM RETAINED COUNTRIES
# ============================================================
quarterly_count_cols = [
    'numerator_base_count', 'numerator_rece_count',
    'tax_count', 'expenditure_count', 'debt_count', 'denominator_count'
]

quarterly_master = (
    df_fpu
    .groupby([
        'iso3',
        pd.Grouper(key='publication_datetime', freq='QE')
    ])[quarterly_count_cols]
    .sum()
    .reset_index()
)

# Raw quarterly indices
quarterly_master['FPU_base_raw'] = (
    quarterly_master['numerator_base_count'] / quarterly_master['denominator_count']
)
quarterly_master['FPU_excl_rec_raw'] = (
    quarterly_master['numerator_rece_count'] / quarterly_master['denominator_count']
)
quarterly_master['Tax_FPU_raw'] = (
    quarterly_master['tax_count'] / quarterly_master['denominator_count']
)
quarterly_master['Expenditure_FPU_raw'] = (
    quarterly_master['expenditure_count'] / quarterly_master['denominator_count']
)
quarterly_master['Debt_FPU_raw'] = (
    quarterly_master['debt_count'] / quarterly_master['denominator_count']
)

quarterly_raw_vars = [
    'FPU_base_raw', 'FPU_excl_rec_raw', 'Tax_FPU_raw',
    'Expenditure_FPU_raw', 'Debt_FPU_raw'
]
quarterly_master[quarterly_raw_vars] = quarterly_master[quarterly_raw_vars].replace(
    [float('inf'), -float('inf')], pd.NA
)

quarterly_norm_map = {
    'FPU_base_raw': 'FPU_base',
    'FPU_excl_rec_raw': 'FPU_excl_rec',
    'Tax_FPU_raw': 'Tax_FPU',
    'Expenditure_FPU_raw': 'Expenditure_FPU',
    'Debt_FPU_raw': 'Debt_FPU'
}
for raw_var, norm_var in quarterly_norm_map.items():
    quarterly_master[norm_var] = (
        quarterly_master.groupby('iso3')[raw_var]
        .transform(normalize_within_country)
    )

quarterly_master['subcategory_sum'] = quarterly_master[
    ['tax_count', 'expenditure_count', 'debt_count']
].sum(axis=1)
quarterly_master['subcategory_sum_minus_aggregate'] = (
    quarterly_master['subcategory_sum'] - quarterly_master['numerator_base_count']
)
quarterly_master['publication_quarter'] = (
    quarterly_master['publication_datetime'].dt.to_period('Q').astype(str)
)
quarterly_master['country'] = cc.convert(names=quarterly_master['iso3'], to='name_short')

quarterly_cols = [
    'iso3', 'country', 'publication_quarter',
    'denominator_count', 'numerator_base_count', 'numerator_rece_count',
    'tax_count', 'expenditure_count', 'debt_count',
    'subcategory_sum', 'subcategory_sum_minus_aggregate',
    'FPU_base_raw', 'FPU_excl_rec_raw',
    'Tax_FPU_raw', 'Expenditure_FPU_raw', 'Debt_FPU_raw',
    'FPU_base', 'FPU_excl_rec', 'Tax_FPU', 'Expenditure_FPU', 'Debt_FPU'
]
quarterly_master = quarterly_master[[c for c in quarterly_cols if c in quarterly_master.columns]]
quarterly_master = quarterly_master.sort_values(['iso3', 'publication_quarter']).reset_index(drop=True)


In [ ]:
# ============================================================
# BUILD GLOBAL MASTER DATASET
# ============================================================
global_master = df_global.copy()
global_master['publication_month'] = (
    pd.to_datetime(global_master['publication_datetime']).dt.to_period('M').astype(str)
)
global_master['subcategory_sum'] = global_master[
    ['tax_count', 'expenditure_count', 'debt_count']
].sum(axis=1)
global_master['subcategory_sum_minus_aggregate'] = (
    global_master['subcategory_sum'] - global_master['numerator_base_count']
)

global_master['FPU_base'] = global_master['FPU_base_transform']
global_master['FPU_excl_rec'] = global_master['FPU_excl_rec_transform']
global_master['Tax_FPU'] = global_master['Tax_FPU_transform']
global_master['Expenditure_FPU'] = global_master['Expenditure_FPU_transform']
global_master['Debt_FPU'] = global_master['Debt_FPU_transform']

global_cols = [
    'publication_month',
    'denominator_count', 'numerator_base_count', 'numerator_rece_count',
    'tax_count', 'expenditure_count', 'debt_count',
    'subcategory_sum', 'subcategory_sum_minus_aggregate',
    'FPU_base_raw', 'FPU_excl_rec_raw',
    'Tax_FPU_raw', 'Expenditure_FPU_raw', 'Debt_FPU_raw',
    'FPU_base', 'FPU_excl_rec', 'Tax_FPU', 'Expenditure_FPU', 'Debt_FPU'
]
global_master = global_master[[c for c in global_cols if c in global_master.columns]]


In [ ]:
# ============================================================
# QC MASTER + FINAL EXCEL EXPORT
# ============================================================
qc_master = monthly_master[[
    'iso3', 'country', 'publication_month',
    'denominator_count', 'numerator_base_count',
    'tax_count', 'expenditure_count', 'debt_count',
    'subcategory_sum', 'subcategory_sum_minus_aggregate'
]].copy()

qc_master['tax_le_aggregate'] = (
    qc_master['tax_count'] <= qc_master['numerator_base_count']
)
qc_master['expenditure_le_aggregate'] = (
    qc_master['expenditure_count'] <= qc_master['numerator_base_count']
)
qc_master['debt_le_aggregate'] = (
    qc_master['debt_count'] <= qc_master['numerator_base_count']
)
qc_master['denominator_positive'] = qc_master['denominator_count'] > 0
qc_master['has_category_overlap'] = qc_master['subcategory_sum_minus_aggregate'] > 0

# Legacy compact sheets retained for downstream compatibility.
fpu_legacy = monthly_master[[
    'iso3', 'country', 'publication_month',
    'FPU_base', 'FPU_excl_rec', 'Tax_FPU', 'Expenditure_FPU', 'Debt_FPU'
]].copy()
gfpu_legacy = global_master[[
    'publication_month',
    'FPU_base', 'FPU_excl_rec', 'Tax_FPU', 'Expenditure_FPU', 'Debt_FPU'
]].copy()

output_dir = f"{cwd}/output"
os.makedirs(output_dir, exist_ok=True)
output_file = f"{output_dir}/FPU_overall_clean.xlsx"

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    monthly_master.to_excel(writer, index=False, sheet_name='Monthly')
    quarterly_master.to_excel(writer, index=False, sheet_name='Quarterly')
    global_master.to_excel(writer, index=False, sheet_name='Global')
    qc_master.to_excel(writer, index=False, sheet_name='QC')
    fpu_legacy.to_excel(writer, index=False, sheet_name='fpu')
    gfpu_legacy.to_excel(writer, index=False, sheet_name='gfpu')

print(f"Saved master FPU workbook: {output_file}")
print("Sheets: Monthly, Quarterly, Global, QC, fpu, gfpu")
print("Monthly master columns:")
print(monthly_master.columns.tolist())
